# Day 27 — Persistent memory (SQLite)

Long-term memory should survive a restart. `sqlite3` (stdlib) is the simplest durable
store. Note the **parameterized query** (`?`) — never f-string user content into SQL
(that's SQL injection, OWASP A03). ✅ Offline, no install.

In [ ]:
# ▶ Run this first — puts the repo root on sys.path so `shared/` imports work.
import sys, pathlib
root = pathlib.Path.cwd()
while root != root.parent and not (root / "shared" / "llm.py").exists():
    root = root.parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print("repo root on sys.path:", root)

## 🔬 Your turn

Fill in the `TODO`s, then run. The solution is below — try first.

In [ ]:
import sqlite3

class SQLiteMemory:
    def __init__(self, path=":memory:"):
        self.db = sqlite3.connect(path)
        self.db.execute("CREATE TABLE IF NOT EXISTS mem (id INTEGER PRIMARY KEY, role TEXT, content TEXT)")

    def save(self, role, content):
        # TODO: INSERT (role, content) using ? placeholders, then self.db.commit()
        raise NotImplementedError

    def recent(self, n=5):
        # TODO: SELECT role, content ORDER BY id DESC LIMIT ?; return rows oldest-first
        raise NotImplementedError

# m = SQLiteMemory(); m.save("user", "hi"); m.save("assistant", "hello")
# print(m.recent())

## 🔒 Solution

One correct way to do it.

In [ ]:
import sqlite3

class SQLiteMemory:
    def __init__(self, path=":memory:"):
        self.db = sqlite3.connect(path)
        self.db.execute("CREATE TABLE IF NOT EXISTS mem (id INTEGER PRIMARY KEY, role TEXT, content TEXT)")
    def save(self, role, content):
        self.db.execute("INSERT INTO mem (role, content) VALUES (?, ?)", (role, content))
        self.db.commit()
    def recent(self, n=5):
        rows = self.db.execute(
            "SELECT role, content FROM mem ORDER BY id DESC LIMIT ?", (n,)
        ).fetchall()
        return list(reversed(rows))

m = SQLiteMemory()
m.save("user", "my name is Sam")
m.save("assistant", "nice to meet you Sam")
print(m.recent())